# Handing Over: What Makes a Repository Someone Else Can Use

You will be handed repositories. That is the job. On your first week somebody will say
*"take a look at Fatimah's churn model, she left in March"* — and what happens in the next ten
minutes decides whether you are useful or whether you are a person asking questions.

This lesson runs that ten minutes from the other side. You are the one leaving. Your work has to
survive the handover **without you in the room** — because in the market you will not be in the
room, and because a graduation project that only runs on your laptop is a graduation project
that nobody can grade twice.

**Learning objectives**

1. Name the six things a stranger needs in the first ten minutes, and say what each one fails to
   guarantee.
2. Write an audit that checks all six mechanically, over any repository, offline.
3. Repair a real handover repository until the audit passes — README, dependencies, paths, data,
   notebook execution order, and secrets.
4. Demonstrate why deleting a leaked credential does not remove it, and state what actually fixes it.
5. Read this repository's own verification gates as a worked example, including the defects that
   shipped *before* each gate existed.

## 🔗 Where this fits

**Builds on:** every earlier lesson of this tooling strand, and it is the last of the six —
01 the shell and the filesystem, 02 git for one and for two, 03 environments and dependencies,
04 reading a traceback, 05 measuring before optimising. Everything below assumes you can already
run a command, make a commit, create an environment and read an error; this lesson is what you do
*with* those five skills when the audience is a stranger who will never meet you.

**Feeds into:** Course 11 (AIAT 125) *Deploying AI Models*, where a repository that a colleague
cannot run is a repository that a container cannot build; and Course 12 (AIAT 126), the Graduation
Project, which is assessed by people who were not there when you wrote it. It is also the
precondition for peer review: you cannot review work you cannot run.

**Sits beside:** this repository's own `Course 04/datasets/DATA.md` and `tools/data.py`, which are
one real answer to the hardest of the six problems — the data story — and `tools/verify/`, which is
the automated half of this lesson, written for the four hundred-plus notebooks in this repository.

## 📰 What actually happens when a stranger runs your notebook

This is measured, not folklore. Pimentel, Murta, Braganholo and Freire collected **1,159,166
unique notebooks from 264,023 GitHub repositories** and tried to run them. Their abstract reports
the result in one sentence:

> out of 863,878 attempted executions of valid notebooks (i.e., notebooks with defined Python
> version and execution order), only 24.11% executed without errors and only 4.03% produced the
> same results.

Read the second number again. **Four percent** of public notebooks reproduce their own stored
outputs. Not four percent of bad notebooks — four percent of all of them, including the ones
attached to papers.

Three more findings from the same paper name the causes this lesson attacks:

- **Execution order.** Of the 912,343 notebooks with unambiguous execution order, **36.36% have
  cells out-of-order**, and **76.90%** have at least one *skip* in the execution counters — a gap
  that means a cell ran and was then edited or deleted, leaving state behind that the file no
  longer explains.
- **Dependencies.** Only **13.72%** (118,483 of 863,878) declared their dependencies at all in a
  `requirements.txt`, `setup.py` or `Pipfile`. And declaring is not the same as working: when the
  authors tried to install those declarations, the failure rate for `requirements.txt` was
  **61.17%**, for `setup.py` **67.55%**, for `Pipfile` **65.20%**.
- **Titles.** Jupyter's default name is `Untitled`, and the paper notes plainly that this
  "discourages users to choose meaningful names".

And the sixth item on the checklist below has its own literature. Meli, McNiece and Reaves scanned
GitHub for nearly six months and reported:

> We find that not only is secret leakage pervasive — affecting over 100,000 repositories— but
> that thousands of new, unique secrets are leaked every day.

**Why this is good news for you.** Every number above is a floor that is trivially cleared by a
person who has been taught to clear it. A graduate whose repositories run on someone else's
machine is, measurably, in a small minority.

## 1  The ten-minute test

Here is what a competent stranger actually does with a repository they have been handed. They do
it in this order, and they stop at the first thing that blocks them.

1. Read the README. *What is this, and how do I run it?*
2. Create an environment and install the dependencies.
3. Open the entry point and run it top to bottom.
4. Watch it look for data.
5. If it works, start reading the code.

Four of those five steps are about **your repository**, not your model. So the checklist has six
gates, and each one has an honest limit — a thing it will happily pass that is still broken.

| # | Gate | Passes when | What it still cannot tell you |
|---|------|-------------|-------------------------------|
| G1 | **README** | A README exists and contains both an *install* command and a *run* command | Whether those commands are correct, or current |
| G2 | **Dependencies** | Every third-party module the code imports appears in a dependency file | Whether the versions are the ones you actually used |
| G3 | **Paths** | No absolute path literals (`/Users/...`, `/home/...`, `C:\...`) anywhere in the code | Whether the relative paths resolve from the directory the reader will actually be in |
| G4 | **Data** | Every data file the code opens is either in the repository or documented in a data file | Whether the documented download still works |
| G5 | **Notebook** | Every notebook declares its kernel and has execution counts `1..n` | Nothing — until you actually execute it. See §5 |
| G6 | **Secrets** | No credential-shaped literals in the files, and a `.gitignore` that covers them | **Nothing about the git history.** See §6 |

All six are static: they read files, and none of them runs your code. That makes them fast enough
to run on every commit. It also makes G5 the one that lies, because a notebook can satisfy every
static check and still die on line one. That is why §5 exists.

**Say this out loud before you continue:** a checklist is not a guarantee. It is a way of failing
fast on the failures somebody has already had. Section 7 shows you five of this repository's own,
all of which shipped before the check that catches them was written.

In [1]:
# WHAT: set up a scratch laboratory OUTSIDE this repository, and work out which Jupyter kernel
#       we are currently running in so the notebooks we build later can declare it honestly.
# WHY:  everything in this lesson creates, breaks, commits to and deletes repositories. None of
#       that may touch the repository you are sitting in, so we build in a temporary directory
#       and assert - not hope - that it is outside the repo before writing a single byte.
import ast
import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import nbformat

# Find the repository root by walking up looking for a marker that only the root has.
HERE = Path.cwd().resolve()
REPO = next((p for p in [HERE, *HERE.parents] if (p / "tools" / "verify").is_dir()), None)

# The laboratory. mkdtemp() gives us a private directory under the system temp area.
LAB = Path(tempfile.mkdtemp(prefix="handover_lab_")).resolve()

# The guard. If this ever fails, nothing below has run yet.
if REPO is not None:
    assert REPO != LAB and REPO not in LAB.parents, f"LAB {LAB} is inside the repository {REPO}"

print("repository root :", REPO)
print("scratch lab     :", LAB)
print("lab inside repo :", REPO is not None and REPO in LAB.parents)

repository root : /Users/abdullah/Downloads/AI Diploma
scratch lab     : /private/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/handover_lab_x3wapqz4
lab inside repo : False


In [2]:
# WHAT: discover the name of the kernelspec that points at THIS Python, and prepare a git
#       environment that cannot read or write your personal git configuration.
# WHY:  (a) a notebook that does not name its kernel opens against whatever Python Jupyter
#       picks - usually the system one, with none of your libraries. We will write a correct
#       kernelspec into the notebooks we repair, so we need the right name.
#       (b) GIT_CONFIG_GLOBAL=/dev/null makes every git command below ignore ~/.gitconfig
#       entirely. Your name, your aliases, your hooks and your signing key are untouched.
def current_kernel_name(default="python3"):
    """Name of the installed kernelspec that launches this exact interpreter."""
    try:
        from jupyter_client.kernelspec import KernelSpecManager
        me = os.path.normpath(sys.executable)
        my_prefix = os.path.normpath(sys.prefix)
        fallback = None
        for name, spec in KernelSpecManager().get_all_specs().items():
            argv = (spec.get("spec") or {}).get("argv") or []
            if not argv:
                continue
            exe = os.path.normpath(argv[0])
            if exe == me:                     # exact interpreter match - certain
                return name
            # Do NOT use Path.resolve() here: every venv's bin/python is a symlink to the same
            # base interpreter, so resolving makes unrelated environments look identical.
            if os.path.normpath(str(Path(exe).parent.parent)) == my_prefix:
                fallback = fallback or name   # same environment, different launcher
        return fallback or default
    except Exception:
        return default


KERNEL = current_kernel_name()
GIT = shutil.which("git")
GIT_ENV = {
    **os.environ,
    "GIT_CONFIG_GLOBAL": os.devnull,   # ignore ~/.gitconfig
    "GIT_CONFIG_SYSTEM": os.devnull,   # ignore /etc/gitconfig
    "GIT_AUTHOR_NAME": "Handover Lab", "GIT_AUTHOR_EMAIL": "lab@example.invalid",
    "GIT_COMMITTER_NAME": "Handover Lab", "GIT_COMMITTER_EMAIL": "lab@example.invalid",
    "GIT_TERMINAL_PROMPT": "0",        # never block waiting for a password
}


def git(repo, *args, quiet=False):
    """Run one git command inside `repo` and return its stdout."""
    if GIT is None:
        return "(git is not installed on this machine)"
    r = subprocess.run([GIT, "-C", str(repo), *args],
                       env=GIT_ENV, capture_output=True, text=True)
    if r.returncode != 0 and not quiet:
        print(f"  git {' '.join(args)} -> exit {r.returncode}: {r.stderr.strip()[:200]}")
    return r.stdout.rstrip()


print("kernel this notebook is running in :", KERNEL)
print("git executable                     :", GIT)

kernel this notebook is running in : ai-diploma
git executable                     : /usr/bin/git


## 2  The repository you were handed

We build one. It is not a caricature: every defect below is one of the six gates failing in the
most ordinary way possible, and each is a defect this repository or the studies above have
actually seen.

The story: a student called Nora finished a Titanic survival analysis last term and left. You have
her folder. Here is what is in it.

In [3]:
# WHAT: create Nora's handover repository inside the scratch lab, with six ordinary defects.
# WHY:  you cannot practise auditing on a repository that is already clean. Every file below is
#       written from this cell, so you can read exactly what the defect is before the audit
#       finds it - and so the lesson never depends on downloading somebody's broken code.
NORA = LAB / "titanic-survival"
(NORA).mkdir(parents=True, exist_ok=True)

# G1 defect: a README that says nothing a stranger can act on.
(NORA / "README.md").write_text(
    "# Titanic survival\n\n"
    "My project for the AI course. Final version.\n\n"
    "Contact: nora\n",
    encoding="utf-8")

# G6 defect: a real-looking credential committed in a source file.
# (The value below is a deliberate fake and is not a credential to anything.)
(NORA / "config.py").write_text(
    '# Settings for the analysis.\n'
    'API_KEY = "tvtc-lab-0000-EXAMPLE-not-a-real-credential"\n'
    'DATA_DIR = "/Users/nora/Desktop/thesis"          # G3 defect: absolute path\n',
    encoding="utf-8")

(NORA / "features.py").write_text(
    '"""Feature helpers."""\n'
    "import numpy as np\n"
    "import pandas as pd\n\n\n"
    "def add_family_size(df):\n"
    '    """SibSp + Parch + the passenger themself."""\n'
    '    out = df.copy()\n'
    '    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1\n'
    "    return out\n",
    encoding="utf-8")

# G3 + G4 + G5 defects, all inside one notebook.
nb = nbformat.v4.new_notebook()
nb.cells = [
    nbformat.v4.new_markdown_cell("# Titanic survival"),
    nbformat.v4.new_code_cell(
        "import pandas as pd\n"
        "from sklearn.linear_model import LogisticRegression\n"
        '# G3/G4: a path that exists on exactly one laptop in the world.\n'
        'df = pd.read_csv("/Users/nora/Desktop/thesis/titanic.csv")\n'
        "print(df.shape)"),
    nbformat.v4.new_code_cell(
        "from features import add_family_size\n"
        "df = add_family_size(df)\n"
        'print(df["FamilySize"].describe())'),
    nbformat.v4.new_code_cell(
        'X = df[["Pclass", "Age", "Fare", "FamilySize"]].fillna(0)\n'
        'y = df["Survived"]\n'
        "model = LogisticRegression(max_iter=1000).fit(X, y)\n"
        'print("training accuracy:", round(model.score(X, y), 3))'),
]
# G5 defect one: the cells were run out of order and the file records it.
for cell, count in zip([c for c in nb.cells if c.cell_type == "code"], [4, 2, 9]):
    cell.execution_count = count
# G5 defect two: no kernelspec at all. Jupyter will guess, and guess wrong.
nb.metadata.pop("kernelspec", None)
nbformat.write(nb, NORA / "analysis.ipynb")

# G2 defect: there is no requirements.txt, no environment.yml, nothing.
# G6 defect: there is no .gitignore either.

for p in sorted(NORA.rglob("*")):
    print(f"{p.stat().st_size:>7} bytes  {p.relative_to(NORA)}")

     80 bytes  README.md
   1212 bytes  analysis.ipynb
    161 bytes  config.py
    230 bytes  features.py


In [4]:
# WHAT: turn Nora's folder into a real git repository with a real commit history.
# WHY:  section 6 needs history to search. Committing inside a throwaway directory under /tmp is
#       safe and is the only way to show, honestly, what a commit does and does not let you undo.
git(NORA, "init", "-q", "-b", "main")
git(NORA, "add", "-A")
git(NORA, "commit", "-q", "-m", "Final version of the analysis")

print(git(NORA, "log", "--oneline"))
print()
print("tracked files:")
print(git(NORA, "ls-files"))

c4d84e0 Final version of the analysis

tracked files:
README.md
analysis.ipynb
config.py
features.py


## 3  The audit

Now we write the checklist as code. Six functions, each returning a verdict and the **evidence**
for it — because "FAIL" without a filename and a line is an accusation, not a finding, and a
reviewer who cannot point at the line has not reviewed anything.

Two helpers come first: one that pulls the code out of a notebook (a `.ipynb` is JSON, so a
plain text search over it also matches outputs, markdown and metadata, and that produces false
positives), and one that works out what a repository actually imports.

In [5]:
# WHAT: helpers - read every code file in a repository as (path, source) pairs, and extract the
#       set of third-party top-level modules the repository imports.
# WHY:  three of the six gates need "the code, and only the code". For a .ipynb that means
#       pulling out the code cells; a grep over the raw JSON would also hit outputs and prose.
DATA_EXT = ("csv", "tsv", "parquet", "json", "jsonl", "xlsx", "xls", "npy", "npz", "pkl", "h5")


def code_sources(repo: Path):
    """Yield (relative_path, python_source) for every .py file and every notebook code cell."""
    for p in sorted(repo.rglob("*.py")):
        if ".git" in p.parts:
            continue
        yield p.relative_to(repo), p.read_text(encoding="utf-8", errors="ignore")
    for p in sorted(repo.rglob("*.ipynb")):
        if ".git" in p.parts or ".ipynb_checkpoints" in p.parts:
            continue
        nb = nbformat.read(p, as_version=4)
        for i, cell in enumerate(nb.cells):
            if cell.cell_type == "code":
                # Strip shell escapes and line magics: they are not Python and break ast.parse.
                src = "\n".join(ln for ln in cell.source.splitlines()
                                if not ln.lstrip().startswith(("!", "%", "%%")))
                yield Path(f"{p.relative_to(repo)}[cell {i}]"), src


def imported_modules(repo: Path):
    """Top-level module names imported by the repository, excluding stdlib and its own modules."""
    # A module is "local" if a file of that name lives anywhere in the repository, or a
    # directory does. src/prep.py is imported as `prep` once src/ is on sys.path.
    local = {q.stem for q in repo.rglob("*.py") if ".git" not in q.parts}
    local |= {d.name for d in repo.rglob("*") if d.is_dir() and ".git" not in d.parts}
    found, unparsed = set(), []
    for rel, src in code_sources(repo):
        try:
            tree = ast.parse(src)
        except SyntaxError:
            unparsed.append(str(rel))       # honest: report it, do not silently drop it
            continue
        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                found.update(a.name.split(".")[0] for a in node.names)
            elif isinstance(node, ast.ImportFrom) and node.level == 0 and node.module:
                found.add(node.module.split(".")[0])
    third_party = {m for m in found if m not in sys.stdlib_module_names and m not in local}
    return sorted(third_party), unparsed


# The import name and the package name are frequently different. This map is HAND-MAINTAINED and
# therefore incomplete - see the limits printed by the audit and discussed in "Where this breaks".
IMPORT_TO_PACKAGE = {
    "sklearn": "scikit-learn", "cv2": "opencv-python", "PIL": "Pillow", "yaml": "PyYAML",
    "bs4": "beautifulsoup4", "skimage": "scikit-image", "dateutil": "python-dateutil",
    "dotenv": "python-dotenv", "serial": "pyserial", "attr": "attrs", "OpenSSL": "pyOpenSSL",
}

mods, unparsed = imported_modules(NORA)
print("third-party modules imported by Nora's repository:", mods)
print("mapped to package names:", [IMPORT_TO_PACKAGE.get(m, m) for m in mods])
print("files ast could not parse:", unparsed or "none")

third-party modules imported by Nora's repository: ['numpy', 'pandas', 'sklearn']
mapped to package names: ['numpy', 'pandas', 'scikit-learn']
files ast could not parse: none


In [6]:
# WHAT: the six gates, each returning (name, passed, [evidence lines]).
# WHY:  a gate that only prints "FAIL" is useless to the person who has to fix it. Every check
#       below returns the file and the text that made it fail, so the report IS the to-do list.
# A Windows path needs a DOUBLED backslash (or a forward slash) after the drive letter,
# because that is how you must write one inside a Python string. Accepting a single
# backslash makes this pattern match "Northpointe's:\n'..." in an ordinary chart label -
# which is exactly what it did the first time it was run over this repository. See
# "Where this breaks".
ABS_PATH = re.compile(r"""['"]((?:/Users/|/home/|/Volumes/|[A-Za-z]:(?:\\\\|/))[^'"\n]*)['"]""")
SECRET_LITERAL = re.compile(
    # The identifier must END with a secret-ish word, so WANDB_API_KEY matches and
    # tokenizer_name does not; the value must be 8+ characters with no whitespace.
    r"""(?i)\b[A-Za-z0-9_]*(?:api[_-]?key|secret|token|password|passwd|credential)"""
    r"""\s*[=:]\s*['"]([^'"\s]{8,})['"]""")
ENV_SECRET = re.compile(r"""(?i)^\s*[A-Z0-9_]*(KEY|SECRET|TOKEN|PASSWORD)[A-Z0-9_]*\s*=\s*\S{8,}""", re.M)
DATA_LITERAL = re.compile(r"""['"]([^'"\n]+\.(?:%s))['"]""" % "|".join(DATA_EXT))
INSTALL_HINT = re.compile(r"(?i)\bpip install\b|\bconda env create\b|\bpoetry install\b|\buv (pip )?(sync|install)\b")
RUN_HINT = re.compile(r"(?i)\bjupyter (lab|notebook)\b|\bpython\s+\S+\.py\b|\bmake\b|\bnbconvert\b|\bstreamlit run\b|\buvicorn\b")
REQ_FILES = ("requirements.txt", "environment.yml", "environment.yaml", "pyproject.toml", "Pipfile")


def _text_files(repo: Path):
    for p in sorted(repo.rglob("*")):
        if not p.is_file() or ".git" in p.parts:
            continue
        if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".gif", ".pdf", ".zip", ".pkl", ".h5"}:
            continue
        yield p.relative_to(repo), p.read_text(encoding="utf-8", errors="ignore")


def g1_readme(repo):
    readme = next((p for p in repo.glob("*") if p.name.lower() in
                   {"readme.md", "readme.rst", "readme.txt", "readme"}), None)
    if readme is None:
        return "G1 README", False, ["no README file at the repository root"]
    txt = readme.read_text(encoding="utf-8", errors="ignore")
    ev = []
    if not INSTALL_HINT.search(txt):
        ev.append(f"{readme.name}: no install command (pip install / conda env create / uv sync)")
    if not RUN_HINT.search(txt):
        ev.append(f"{readme.name}: no run command (jupyter lab / python x.py / make ...)")
    return "G1 README", not ev, ev


def g2_dependencies(repo):
    present = [f for f in REQ_FILES if (repo / f).exists()]
    mods, _ = imported_modules(repo)
    wanted = {IMPORT_TO_PACKAGE.get(m, m).lower() for m in mods}
    if not present:
        return "G2 dependencies", False, [
            f"no dependency file ({', '.join(REQ_FILES)})",
            f"code imports: {', '.join(sorted(wanted)) or 'nothing third-party'}"]
    declared = set()
    for f in present:
        for line in (repo / f).read_text(encoding="utf-8", errors="ignore").splitlines():
            line = line.split("#")[0].strip().lstrip("- ").strip()
            if not line:
                continue
            name = re.split(r"[<>=!~\[; ]", line, maxsplit=1)[0].strip().lower()
            if name and not name.endswith(":"):
                declared.add(name)
    missing = sorted(wanted - declared)
    ev = [f"imported but not declared in {'/'.join(present)}: {m}" for m in missing]
    return "G2 dependencies", not ev, ev


def g3_paths(repo):
    ev = []
    for rel, src in code_sources(repo):
        for m in ABS_PATH.finditer(src):
            ev.append(f"{rel}: {m.group(1)}")
    return "G3 paths", not ev, ev


def g4_data(repo):
    docs = " ".join(txt for rel, txt in _text_files(repo)
                    if rel.suffix.lower() in {".md", ".rst", ".txt"})
    ev = []
    for rel, src in code_sources(repo):
        for m in DATA_LITERAL.finditer(src):
            ref = m.group(1)
            if (repo / ref.lstrip("./")).exists():
                continue
            if Path(ref).name in docs:
                continue                     # missing on disk, but the docs say where to get it
            ev.append(f"{rel}: opens {ref!r}, which is neither in the repository nor documented")
    return "G4 data", not ev, ev


def g5_notebooks(repo):
    ev = []
    for p in sorted(repo.rglob("*.ipynb")):
        if ".git" in p.parts or ".ipynb_checkpoints" in p.parts:
            continue
        rel = p.relative_to(repo)
        nb = nbformat.read(p, as_version=4)
        if not (nb.metadata.get("kernelspec") or {}).get("name"):
            ev.append(f"{rel}: declares no kernel, so Jupyter will pick one for the reader")
        counts = [c.get("execution_count") for c in nb.cells if c.cell_type == "code"]
        if counts and all(c is None for c in counts):
            ev.append(f"{rel}: no cell has been executed - run Restart and Run All before handing over")
        elif counts and counts != list(range(1, len(counts) + 1)):
            ev.append(f"{rel}: execution counts are {counts}, not {list(range(1, len(counts) + 1))}")
    return "G5 notebooks", not ev, ev


def g6_secrets(repo):
    ev = []
    # Notebooks are JSON, so their quotes arrive escaped as \" and a plain text search over the
    # file misses every literal inside a code cell. Read the CODE, the way G3 does.
    for rel, src in code_sources(repo):
        for m in SECRET_LITERAL.finditer(src):
            ev.append(f"{rel}: {m.group(0).strip()[:70]}")
    for rel, txt in _text_files(repo):
        if rel.suffix.lower() in {".py", ".ipynb"}:
            continue                         # already covered by code_sources, above
        pattern = ENV_SECRET if rel.name.startswith(".env") else SECRET_LITERAL
        for m in pattern.finditer(txt):
            ev.append(f"{rel}: {m.group(0).strip()[:70]}")
    gitignore = repo / ".gitignore"
    if not gitignore.exists():
        ev.append("no .gitignore, so a future .env or credentials file gets committed by accident")
    elif ".env" not in gitignore.read_text(encoding="utf-8", errors="ignore"):
        ev.append(".gitignore does not mention .env")
    return "G6 secrets", not ev, ev


GATES = [g1_readme, g2_dependencies, g3_paths, g4_data, g5_notebooks, g6_secrets]


def audit(repo: Path, title=""):
    """Run all six gates and print a report. Returns the number that passed."""
    print(f"AUDIT  {title or repo.name}")
    print("=" * 72)
    passed = 0
    for gate in GATES:
        name, ok, evidence = gate(repo)
        passed += ok
        print(f"{'PASS' if ok else 'FAIL'}  {name}")
        for line in evidence[:6]:
            print(f"        - {line}")
        if len(evidence) > 6:
            print(f"        - ... and {len(evidence) - 6} more")
    print("=" * 72)
    print(f"{passed} of {len(GATES)} gates pass\n")
    return passed

In [7]:
# WHAT: run the audit on the repository exactly as it was handed to us.
# WHY:  this is the report you would send back to Nora, or paste into a review. Read it as a
#       to-do list: every line names a file and the text that made the gate fail.
before = audit(NORA, "titanic-survival, as received")

AUDIT  titanic-survival, as received
FAIL  G1 README
        - README.md: no install command (pip install / conda env create / uv sync)
        - README.md: no run command (jupyter lab / python x.py / make ...)
FAIL  G2 dependencies
        - no dependency file (requirements.txt, environment.yml, environment.yaml, pyproject.toml, Pipfile)
        - code imports: numpy, pandas, scikit-learn
FAIL  G3 paths
        - config.py: /Users/nora/Desktop/thesis
        - analysis.ipynb[cell 1]: /Users/nora/Desktop/thesis/titanic.csv
FAIL  G4 data
        - analysis.ipynb[cell 1]: opens '/Users/nora/Desktop/thesis/titanic.csv', which is neither in the repository nor documented
FAIL  G5 notebooks
        - analysis.ipynb: declares no kernel, so Jupyter will pick one for the reader
        - analysis.ipynb: execution counts are [4, 2, 9], not [1, 2, 3]
FAIL  G6 secrets
        - config.py: API_KEY = "tvtc-lab-0000-EXAMPLE-not-a-real-credential"
        - no .gitignore, so a future .env or credentia

**Read the report.** Six gates, six failures, and not one of them is about machine learning. The
model is fine. Nobody can get to it.

Notice what the evidence gives you that a bare "FAIL" would not: `analysis.ipynb[cell 1]` names the
cell, `/Users/nora/Desktop/thesis/titanic.csv` names the exact literal, and
`execution counts are [4, 2, 9]` tells you the cells were run in the order 2, 1, 3 with five
executions missing in between — which is Pimentel's *skip*, and the reason you cannot trust the
stored outputs.

One more thing to notice, because it is the point of the whole strand: **not one of those findings
required running the code.** Six defects, found by reading files, in under a second. The person who
runs this before sending you the repository has already saved you the ten minutes — and §5 is about
the one finding this method can never make.

## 4  Fix it

We now repair all six, in the order a person actually would. Read each fix as a *pattern* — the
specific commands change, the shape does not.

The data fix deserves a sentence of its own. Nora's notebook reads a path that exists on one
laptop. There are exactly three honest endings to that sentence:

1. **The file is small and shareable** → commit it, and write down where it came from.
2. **The file is large or licensed** → do not commit it; commit a *loader* that finds it, plus a
   document saying where to get it. That is what this repository does: see `tools/data.py` and
   `Course 04/datasets/DATA.md`, which ships a small sample of each big dataset and prints one
   line saying whether you got the sample or the full file.
3. **The file cannot be shared at all** → say so in the README, and ship a synthetic fixture so
   the code path is still runnable. Then the reader knows the number they see is not your number.

Titanic is case 1: 60 KB of real, public data that already ships in this repository. We copy it in
and document its provenance.

In [8]:
# WHAT: repair G1, G2, G3, G4 and G6. G5 is repaired here only in part - see section 5.
# WHY:  each fix below is the minimum that makes the gate pass for a real reason, not the
#       minimum that makes the checker shut up. Those are different, and the difference is
#       the whole subject of this lesson.
import importlib.metadata as md

# --- Keep an untouched copy of the broken notebook, OUTSIDE the repo, for section 5. ----------
BROKEN_COPY = LAB / "analysis_as_received.ipynb"
shutil.copy2(NORA / "analysis.ipynb", BROKEN_COPY)

# --- G4 first: the data has to exist before the path can point at it. -------------------------
if REPO is None or not (REPO / "Course 04" / "datasets" / "raw" / "titanic.csv").exists():
    raise RuntimeError(
        "This lesson copies the real Titanic file that ships in this repository at "
        "'Course 04/datasets/raw/titanic.csv'. It was not found, which means this notebook is "
        "running outside a clone of the AI Diploma repository. Clone the repository and re-run.")
(NORA / "data").mkdir(exist_ok=True)
shutil.copy2(REPO / "Course 04" / "datasets" / "raw" / "titanic.csv", NORA / "data" / "titanic.csv")
(NORA / "data" / "DATA.md").write_text(
    "# Data\n\n"
    "## titanic.csv\n\n"
    "- **What it is:** the Titanic passenger manifest, 891 rows, one row per passenger.\n"
    "- **Where it came from:** the copy that ships with the AI Diploma repository at\n"
    "  `Course 04/datasets/raw/titanic.csv`. It is small (about 60 KB) and public, so it is\n"
    "  committed here rather than downloaded.\n"
    "- **Licence / restrictions:** public dataset, no restrictions on classroom use.\n"
    "- **How the code reads it:** `pd.read_csv('data/titanic.csv')`, relative to the repository\n"
    "  root, which is the directory Jupyter uses when you open `analysis.ipynb`.\n",
    encoding="utf-8")

# --- G3 + part of G5: rewrite the notebook's path and clear its stale execution state. --------
nb = nbformat.read(NORA / "analysis.ipynb", as_version=4)
for cell in nb.cells:
    if cell.cell_type == "code":
        cell.source = cell.source.replace(
            '"/Users/nora/Desktop/thesis/titanic.csv"', '"data/titanic.csv"')
        cell.source = cell.source.replace(
            "# G3/G4: a path that exists on exactly one laptop in the world.\n",
            "# Relative to the repository root - works on any machine that cloned it.\n")
        cell.execution_count = None          # the old counts described a session nobody has
        cell.outputs = []                    # and outputs nobody can reproduce
nb.metadata["kernelspec"] = {"display_name": f"Python ({KERNEL})", "language": "python",
                             "name": KERNEL}
nb.nbformat_minor = max(nb.nbformat_minor, 5)
nbformat.write(nb, NORA / "analysis.ipynb")

# --- G3 again: the same literal is in config.py. ----------------------------------------------
# --- G6: and so is the credential. Both go at once. -------------------------------------------
(NORA / "config.py").write_text(
    '"""Settings. Secrets come from the environment, never from this file."""\n'
    "import os\n\n"
    "# Read at import time; the error message tells the reader exactly what to do.\n"
    'API_KEY = os.environ.get("TITANIC_API_KEY", "")\n'
    'DATA_DIR = "data"          # relative to the repository root\n',
    encoding="utf-8")
(NORA / ".env.example").write_text(
    "# Copy to .env and fill in. .env is gitignored and must never be committed.\n"
    "TITANIC_API_KEY=\n",
    encoding="utf-8")
(NORA / ".gitignore").write_text(
    "# Secrets\n.env\n*.pem\n*.key\n\n"
    "# Python\n__pycache__/\n*.py[cod]\n.venv/\nvenv/\n\n"
    "# Jupyter\n.ipynb_checkpoints/\n",
    encoding="utf-8")

# --- G2: write a dependency file from what the code actually imports, pinned to what is
#         installed right now. Guessing the versions afterwards is how the 61.17% happens. ------
mods, _ = imported_modules(NORA)
lines = []
for m in mods:
    pkg = IMPORT_TO_PACKAGE.get(m, m)
    try:
        lines.append(f"{pkg}=={md.version(pkg)}")
    except md.PackageNotFoundError:
        lines.append(pkg)                    # installed under another name: flag, do not invent
(NORA / "requirements.txt").write_text(
    "# Generated from the imports this repository actually makes, pinned to the versions it\n"
    "# was last run with. Regenerate after adding an import.\n" + "\n".join(lines) + "\n",
    encoding="utf-8")

# --- G1: a README a stranger can act on. ------------------------------------------------------
(NORA / "README.md").write_text(f"""# Titanic survival

Logistic regression predicting passenger survival from class, age, fare and family size.
One notebook, one dataset, about ten seconds end to end.

## Install

```bash
python3 -m venv .venv
source .venv/bin/activate          # Windows: .venv\\Scripts\\activate
pip install -r requirements.txt
```

## Run

```bash
jupyter lab analysis.ipynb         # then Kernel -> Restart Kernel and Run All Cells
```

The notebook is verified against the `{KERNEL}` kernel. Select it if Jupyter offers a choice.

## Data

`data/titanic.csv` ships with this repository. Its provenance is in `data/DATA.md`.

## Configuration

Copy `.env.example` to `.env` if you need to set `TITANIC_API_KEY`. `.env` is gitignored.

## What you should see

`analysis.ipynb` prints the dataset shape, a summary of the engineered `FamilySize` column,
and the training accuracy of the fitted model.
""", encoding="utf-8")

print("requirements.txt now contains:")
print((NORA / "requirements.txt").read_text(encoding="utf-8"))
print("files after the repair:")
for p in sorted(NORA.rglob("*")):
    if ".git" not in p.parts and p.is_file():
        print("   ", p.relative_to(NORA))

requirements.txt now contains:
# Generated from the imports this repository actually makes, pinned to the versions it
# was last run with. Regenerate after adding an import.
numpy==2.4.4
pandas==2.3.3
scikit-learn==1.8.0

files after the repair:
    .env.example
    .gitignore
    README.md
    analysis.ipynb
    config.py
    data/DATA.md
    data/titanic.csv
    features.py
    requirements.txt


## 5  The only check that cannot be faked: Restart and Run All

Every gate so far read files. G5 read the *record* of an execution — the kernel name and the
counters — and a record can be tidy and still be a lie. Pimentel's four percent is exactly the gap
between a file that looks executed and a file that executes.

So we execute both notebooks, in a fresh kernel, with the repository as the working directory,
which is what `Restart Kernel and Run All Cells` does in Jupyter and what
`jupyter nbconvert --execute` does on a build server. The one you were handed should fail. The
repaired one should not.

In [9]:
# WHAT: execute a notebook in a brand-new kernel and report honestly what happened.
# WHY:  this is Restart-and-Run-All, expressed as code. nbclient starts a fresh kernel, so no
#       variable from this session leaks in - which is the entire point. `resources` sets the
#       working directory, exactly as Jupyter sets it to the notebook's own folder.
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError


def restart_and_run_all(path: Path, cwd: Path, kernel=None, write_back=False):
    """Return (ok, message). Executes `path` from scratch with `cwd` as the working directory."""
    nb = nbformat.read(path, as_version=4)
    name = kernel or (nb.metadata.get("kernelspec") or {}).get("name") or KERNEL
    client = NotebookClient(nb, timeout=120, kernel_name=name,
                            resources={"metadata": {"path": str(cwd)}})
    try:
        client.execute()
    except CellExecutionError as exc:
        last = [ln for ln in str(exc).strip().splitlines() if ln.strip()][-1]
        return False, f"{type(exc).__name__}: {last.strip()}"
    if write_back:
        nbformat.write(nb, path)
    outs = [o.get("text", "") for c in nb.cells if c.cell_type == "code"
            for o in c.get("outputs", []) if o.get("output_type") == "stream"]
    return True, "".join(outs).strip()


ok_before, msg_before = restart_and_run_all(BROKEN_COPY, cwd=LAB)
print("as received :", "RAN CLEAN" if ok_before else "FAILED")
print("   ", msg_before[:300])
print()

ok_after, msg_after = restart_and_run_all(NORA / "analysis.ipynb", cwd=NORA, write_back=True)
print("after repair:", "RAN CLEAN" if ok_after else "FAILED")
for line in msg_after.splitlines():
    print("   ", line)

as received : FAILED
    CellExecutionError: FileNotFoundError: [Errno 2] No such file or directory: '/Users/nora/Desktop/thesis/titanic.csv'



after repair: RAN CLEAN
    (891, 12)
    count    891.000000
    mean       1.904602
    std        1.613459
    min        1.000000
    25%        1.000000
    50%        1.000000
    75%        2.000000
    max       11.000000
    Name: FamilySize, dtype: float64
    training accuracy: 0.705


In [10]:
# WHAT: re-run the full audit now that the notebook has genuinely executed top to bottom.
# WHY:  G5's execution counts only become 1..n because a real kernel really ran every cell in
#       order. The gate now reports something that is true rather than something that was typed.
after = audit(NORA, "titanic-survival, after the repair")
print(f"gates passing: {before} -> {after}")

AUDIT  titanic-survival, after the repair
PASS  G1 README
PASS  G2 dependencies
PASS  G3 paths
PASS  G4 data
PASS  G5 notebooks
PASS  G6 secrets
6 of 6 gates pass

gates passing: 0 -> 6


**What just happened, precisely.** The notebook you were handed did not fail on the model, the
maths, or the data science. It failed on `FileNotFoundError`, at the first line that touched the
outside world. That is the modal failure of a handed-over repository, and the repair was one line.

And the execution counts in the repaired notebook are `1, 2, 3` because a kernel really produced
them, in that order, one after another, from nothing. You did not renumber them. **Never renumber
them.** The counter is evidence; editing evidence to satisfy a checker is the exact behaviour a
checklist is supposed to prevent.

## 6  The gate that passes while the repository is still compromised

G6 now says PASS. The credential is gone from `config.py`, `.gitignore` covers `.env`, and any
scanner reading the working tree agrees the repository is clean.

It is not clean. Commit the deletion and look in the history.

In [11]:
# WHAT: commit the repair, then search the ENTIRE history - not the working tree - for the
#       credential we just removed.
# WHY:  `git commit` adds; it does not subtract. Every earlier version of every file is still
#       reachable, which is the property that makes git useful and makes leaked secrets permanent.
git(NORA, "add", "-A")
git(NORA, "commit", "-q", "-m", "Repair the handover: README, requirements, paths, data, secrets")

print("history now:")
print(git(NORA, "log", "--oneline"))
print()
print("does the credential appear in the working tree?")
tree_hit = git(NORA, "grep", "-n", "tvtc-lab-0000", quiet=True)
print("   ", tree_hit or "NO - the working tree is clean")
print()

revs = git(NORA, "rev-list", "--all").split()
print(f"searching {len(revs)} commit(s) of history for the same string...")
hist_hit = git(NORA, "grep", "-n", "tvtc-lab-0000", *revs, quiet=True) if revs else ""
print(hist_hit or "   nothing found in history")

history now:
1bb65e5 Repair the handover: README, requirements, paths, data, secrets
c4d84e0 Final version of the analysis

does the credential appear in the working tree?
    NO - the working tree is clean

searching 2 commit(s) of history for the same string...
c4d84e03f92c13d08005fc37df2c0b4d62d7ef93:config.py:2:API_KEY = "tvtc-lab-0000-EXAMPLE-not-a-real-credential"


**There it is.** `git grep` across `git rev-list --all` finds the credential in the first commit,
in a file that no longer exists, in a repository whose latest version passes every static check.
Anyone who has ever cloned this repository has that value on their disk right now. Anyone who
clones it in five years still will.

**So what actually fixes it?** In order of what matters:

1. **Rotate the credential.** Immediately. Revoke the old one at the provider and issue a new one.
   This is the only step that removes the risk, and it is the only step most people skip. Meli et
   al. scanned GitHub for months precisely because attackers do the same scan, continuously.
2. **Then** decide whether to rewrite history (`git filter-repo`, or the provider's secret-removal
   support). It is disruptive: every commit hash after the rewrite changes, so everyone with a
   clone has to re-clone. Do it for a private repository you control. Understand that for a public
   repository the value was already copied before you finished reading this sentence.
3. Add the pattern to `.gitignore` and to a pre-commit scan, so the next one never reaches a commit.

**The transferable idea, which is the point of this whole section:** *a gate tells you the truth
about what it looked at.* G6 looked at the working tree, reported honestly about the working tree,
and was completely uninformative about the risk. When you are handed a green checklist, your first
question is not "did it pass" — it is **"what did it read?"**

## 7  A worked example with real stakes: this repository's own gates

This repository is a handover artefact. Twelve courses, hundreds of notebooks, written to be run
by students on machines nobody has seen, and by an instructor who has to grade the same lesson
twice a year. It carries the automated half of everything above in `tools/verify/`.

Run the same four static checks over the twelve-course student path — a bigger, real repository
instead of a toy — and see what they say.

In [12]:
# WHAT: run four of the six checks over every notebook on this repository's student path.
# WHY:  an audit you have only ever run on a repository you built yourself has not been tested.
#       This is 420 real notebooks written over months by several hands. Read-only: this cell
#       opens files and writes nothing.
import json

if REPO is None:
    print("Not inside a clone of the repository - skipping.")
else:
    nbs = sorted(REPO.glob("Course */unit*/*/*.ipynb"))
    no_kernel, out_of_order, abs_paths, secrets, unreadable = [], [], [], [], []
    for p in nbs:
        try:
            doc = json.loads(p.read_text(encoding="utf-8"))
        except Exception as exc:
            unreadable.append((p.name, type(exc).__name__))
            continue
        rel = p.relative_to(REPO)
        if not ((doc.get("metadata") or {}).get("kernelspec") or {}).get("name"):
            no_kernel.append(str(rel))
        counts = [c.get("execution_count") for c in doc.get("cells", [])
                  if c.get("cell_type") == "code"]
        counts = [c for c in counts if c is not None]
        if counts and counts != list(range(1, len(counts) + 1)):
            out_of_order.append((str(rel), counts[:6]))
        for c in doc.get("cells", []):
            if c.get("cell_type") != "code":
                continue
            src = "".join(c.get("source", []))
            if ABS_PATH.search(src):
                abs_paths.append((str(rel), ABS_PATH.search(src).group(1)))
            if SECRET_LITERAL.search(src):
                secrets.append((str(rel), SECRET_LITERAL.search(src).group(0)[:60]))

    print(f"notebooks scanned                 : {len(nbs)}")
    print(f"unreadable (JSON parse failure)   : {len(unreadable)}")
    print(f"G5a  no kernel declared           : {len(no_kernel)}")
    print(f"G5b  counts not 1..n              : {len(out_of_order)}   (cleared cells ignored)")
    print(f"G3   absolute path in a code cell : {len(abs_paths)}")
    print(f"G6   credential-shaped literal    : {len(secrets)}")
    for label, items in [("no kernel", no_kernel), ("out of order", out_of_order),
                         ("absolute path", abs_paths), ("secret-shaped", secrets)]:
        for item in items[:5]:
            print(f"   {label}: {item}")

notebooks scanned                 : 420
unreadable (JSON parse failure)   : 0
G5a  no kernel declared           : 0
G5b  counts not 1..n              : 0   (cleared cells ignored)
G3   absolute path in a code cell : 0
G6   credential-shaped literal    : 0


In [13]:
# WHAT: print the real commit history of this repository's verification tooling.
# WHY:  the dates are the lesson. Read when each gate was added, and ask what the repository
#       looked like before that date - because the answer is "it had the defect".
if REPO is None or GIT is None:
    print("No repository or no git - skipping. (A .zip download has no history to read.)")
else:
    log = git(REPO, "log", "--date=short", "--format=%ad  %s", "--", "tools/verify")
    if not log:
        print("No history for tools/verify - is this a shallow or zip copy?")
    else:
        print("history of tools/verify (oldest last):")
        for line in log.splitlines():
            print("  ", line)
        print()
        scripts = sorted(q.name for q in (REPO / "tools" / "verify").glob("*.py"))
        print(f"{len(scripts)} python files in tools/verify today:")
        for name in scripts:
            print("   ", name)
        print()
        # Print two of those commit messages in full, so the claims made below this cell can be
        # checked against the repository rather than taken on trust.
        for needle in ("every notebook now declares the kernel",
                       "two gates for the defect classes"):
            body = git(REPO, "log", "-1", "--format=%B", f"--grep={needle}", quiet=True)
            print("-" * 72)
            for line in (body or "(commit not found in this clone)").splitlines()[:14]:
                print(line)

history of tools/verify (oldest last):
   2026-09-01  fix(verify): the leak detector was blind to every code-writing answer - verb list removed
   2026-09-01  feat(verify): two gates for the defect classes that slipped past every existing check
   2026-08-31  fix(portability): every notebook now declares the kernel it was verified on
   2026-08-29  fix(consistency): settle 20 cross-course contradictions and add docs/GLOSSARY.md
   2026-08-29  feat(frontier): 2024-2026 citations + state-of-field boxes for Courses 02 and 04
   2026-08-25  feat(real-data): last 10 fabrications converted - fabricating notebooks now 11/403, all legitimate
   2026-08-24  feat(verify): answer-key gate now flags unbalanced code fences in quizzes
   2026-08-23  chore: parse baseline refreshed (0/404)
   2026-08-20  feat(partc/course01): author 10 critical gap notebooks — U3 GD+SHAP/LIME, U4 five Keras labs + exercise, U5 toy GAN
   2026-08-20  feat(phase6/root): root README + docs truth pass, timeline/map retir

### What that history says, and it is not flattering

Read the dates against the previous cell's four gate results, every one of them zero. **The
repository is clean today — and it was not clean before the gate that catches each defect was
written.** In every case the gate came *after* the defect had already shipped. Five of them, taken
from the commit messages you just printed:

- **`2026-08-31`, kernel declarations.** The check that printed `G5a  no kernel declared : 0`
  above did not exist until the last week of August. Its commit records what it found on the day it was
  first run: *"343 of 420 notebooks (82%) did not name their kernel"* — 218 of them declaring
  `python3`, which on that machine resolved to Xcode's Python 3.9.6, with no `shap`, no `lime`,
  no TensorFlow. The sharpest case was Course 01's SHAP/LIME lesson, which imports both libraries
  and named no kernel at all: *"it could only ever have worked for someone who happened to select
  the right kernel by hand."* That is gate G5 finding a day-one blocker for **every** student, in
  a repository that had passed every other check for months.

- **`2026-08-31`, the runner's own blind spot.** The same commit fixes
  `tools/verify/run_notebooks.py`, whose file glob covered `examples/` and `exercises/` but not
  `enrichment/`. Seventeen lessons were *invisible to every course-level verification run ever
  performed*. The gate was green because the gate was not looking.

- **`2026-08-24`, unbalanced code fences.** The answer-key gate searched for marker text such as
  `**Answer:**`. Five quizzes had solutions visible to students anyway, exposed by a code fence
  that was opened and never closed — the marker was never there to find. The fix counts the fences
  in each file and flags an odd number.

- **`2026-09-01`, the leak detector that matched on wording.** Its first version fired only when a
  question matched a list of prompt verbs. It missed 11 of 26 known leaks in Course 03 and found
  nothing at all in Course 02. Rewritten to judge *structure* rather than wording, it immediately
  found two more real leaks in Course 02 that the previous version had reported as clean.

- **`2026-09-01`, two gates for defects nothing was looking for.** `check_key_rubric_agreement.py`
  compares an exam's answer key against its rubric; five courses disagreed with themselves. Its
  commit records the worst: in Course 04, *"a fully correct Part 1 scored 25 of 30 wrong by anyone
  grading from the rubric."*

There is a second, quieter admission in that last commit worth copying into your own habits:

> Both gates were themselves debugged against false positives before being trusted: the first
> flagged a search course's node-visit order 'Answer: A, B, C, D' as an answer key, the second
> missed every multiple-choice block because its option regex lacked `re.MULTILINE`.

**Three conclusions, and they are the ones that transfer.**

1. Every gate is a fossil of a real failure. Nobody writes a check for a defect they have never
   had. When you join a team, read their CI configuration as a list of things that once went wrong.
2. A green check bounds your ignorance; it does not remove it. The four zeros above are worth
   exactly the coverage of the four checks and no more.
3. **Check the checker.** A gate that has never fired is either protecting you or not looking at
   anything, and from the outside those are identical. Break something on purpose and confirm the
   gate catches it — which is precisely what section 5 did with a deliberately broken notebook.

## 8  Your turn

Everything above was demonstration. This part is yours, and there is no worked answer for it
anywhere in this notebook.

A second repository is built below: **`survival-baseline`**, handed over by a different student. Its
defects are *not* the same as Nora's — some gates that failed for her pass here, and the ones that
fail, fail differently. Run the audit, read the evidence, and repair the repository until all six
gates pass **for real reasons**.

### The task

Write your fixes inside `fix_survival_baseline(repo)` in the cell after next, then re-run the audit cell.

### The rules — these are what you would be judged on in a review

1. **Fix the cause, not the checker.** Deleting a line so a regex stops matching is not a repair.
   Every gate below is satisfiable dishonestly; a reviewer will read your diff, not your score.
2. **The data decision is yours to justify.** The CSV `survival-baseline` needs is sitting in the lab
   folder at `LAB / "downloads" / "titanic.csv"` — the way a file sits in your Downloads folder,
   outside the repository, invisible to anyone you send the repository to. Pick one of the three
   endings from section 4 and make the repository tell the truth about which one you picked.
3. **Never renumber execution counts by hand.** Clear them, then execute the notebook with
   `restart_and_run_all(..., write_back=True)`. If it does not run, that is a finding, not an
   obstacle.
4. **Rotate, then remove.** The token in this repository would, if real, need revoking *before*
   anything else. Write that instruction where the next reader will see it.
5. **Leave the repository better than the audit requires.** Two of the six gates can be passed by a
   file that is technically present and useless to a human. You know which two.

### How you know you are finished

- `audit(BASELINE)` prints **6 of 6 gates pass**, and
- `restart_and_run_all(BASELINE / "train.ipynb", cwd=BASELINE)` returns `True`, and
- a classmate who has never seen your repository can follow your README from a clean shell and get
  the same numbers.

The third one is the real test, and it is the one with evidence behind it. Double, McGrane and
Hopfenbeck's meta-analysis of 54 studies found that peer assessment improved academic performance
with an overall effect of *g* = 0.31, and — this is the part people find surprising — slightly
more than teacher assessment did (*g* = 0.28). Read that with its limits attached: the corpus is
mostly writing and general coursework rather than code, and the studies are largely
quasi-experimental. It is a reason to swap repositories with a classmate, not a guarantee.

So swap. Audit theirs, and hand back **evidence lines, not a verdict** — the file, the line, and
what a stranger would hit. "Your README is bad" is worth nothing to the person who has to fix it.

In [14]:
# WHAT: build the second handover repository and audit it as received.
# WHY:  this is your exercise. Read the report before you read the files - that is the order a
#       reviewer works in, and it stops you fixing things nobody asked about.
BASELINE = LAB / "survival-baseline"
(BASELINE / "src").mkdir(parents=True, exist_ok=True)
(LAB / "downloads").mkdir(exist_ok=True)

# The dataset is on "your laptop", outside the repository. Same real file, different problem.
shutil.copy2(REPO / "Course 04" / "datasets" / "raw" / "titanic.csv",
             LAB / "downloads" / "titanic.csv")

(BASELINE / "README.md").write_text(
    "# Survival baseline\n\n"
    "Random-forest baseline on the Titanic passenger manifest.\n\n"
    "## Install\n\n"
    "```bash\n"
    "pip install -r requirements.txt\n"
    "```\n",
    encoding="utf-8")

# A dependency file that exists and is wrong - which the 61.17% installation-failure rate is made of.
(BASELINE / "requirements.txt").write_text("pandas\n", encoding="utf-8")

(BASELINE / "src" / "prep.py").write_text(
    '"""Preparation helpers."""\n'
    "import numpy as np\n"
    "import pandas as pd\n\n\n"
    "def to_features(df):\n"
    '    out = df[["Pclass", "Age", "Fare"]].copy()\n'
    '    out["Age"] = out["Age"].fillna(out["Age"].median())\n'
    "    return out\n",
    encoding="utf-8")

nb = nbformat.v4.new_notebook()
nb.cells = [
    nbformat.v4.new_markdown_cell("# Survival baseline"),
    nbformat.v4.new_code_cell(
        "import sys\n"
        "import pandas as pd\n"
        "from sklearn.ensemble import RandomForestClassifier\n"
        'sys.path.insert(0, "src")\n'
        "from prep import to_features\n"
        'WANDB_API_KEY = "tvtc-lab-1111-EXAMPLE-not-a-real-credential"   # tracking token'),
    nbformat.v4.new_code_cell(
        'df = pd.read_csv("data/titanic.csv")\n'
        "X, y = to_features(df), df[\"Survived\"]\n"
        "print(X.shape)"),
    nbformat.v4.new_code_cell(
        "model = RandomForestClassifier(n_estimators=50, random_state=0).fit(X, y)\n"
        'print("training accuracy:", round(model.score(X, y), 3))'),
]
for cell, count in zip([c for c in nb.cells if c.cell_type == "code"], [1, 2, 5]):
    cell.execution_count = count
nb.metadata["kernelspec"] = {"display_name": f"Python ({KERNEL})", "language": "python",
                             "name": KERNEL}
nbformat.write(nb, BASELINE / "train.ipynb")

if GIT:
    git(BASELINE, "init", "-q", "-b", "main")
    git(BASELINE, "add", "-A")
    git(BASELINE, "commit", "-q", "-m", "baseline v1")

print("files you were handed:")
for p in sorted(BASELINE.rglob("*")):
    if ".git" not in p.parts and p.is_file():
        print("   ", p.relative_to(BASELINE))
print("the dataset, sitting outside the repository:", (LAB / "downloads" / "titanic.csv").name)
print()
_ = audit(BASELINE, "survival-baseline, as received")

files you were handed:
    README.md
    requirements.txt
    src/prep.py
    train.ipynb
the dataset, sitting outside the repository: titanic.csv

AUDIT  survival-baseline, as received
FAIL  G1 README
        - README.md: no run command (jupyter lab / python x.py / make ...)
FAIL  G2 dependencies
        - imported but not declared in requirements.txt: numpy
        - imported but not declared in requirements.txt: scikit-learn
PASS  G3 paths
FAIL  G4 data
        - train.ipynb[cell 2]: opens 'data/titanic.csv', which is neither in the repository nor documented
FAIL  G5 notebooks
        - train.ipynb: execution counts are [1, 2, 5], not [1, 2, 3]
FAIL  G6 secrets
        - train.ipynb[cell 1]: WANDB_API_KEY = "tvtc-lab-1111-EXAMPLE-not-a-real-credential"
        - no .gitignore, so a future .env or credentials file gets committed by accident
1 of 6 gates pass



In [15]:
# WHAT: YOUR WORK. Repair survival-baseline so that all six gates pass for real reasons.
# WHY:  the audit is the judge, but the audit is not the point - a person who clones this
#       repository and follows your README is. Write the fixes, then run the cell below.
def fix_survival_baseline(repo: Path) -> None:
    """Repair the repository at `repo`. Delete the `pass` and write your fixes here.

    Suggested order - the same order section 4 used, and for the same reasons:
      1. Data first, because the path cannot point at a file that does not exist.
         The file is at LAB / "downloads" / "titanic.csv". Decide: commit it, or document it.
      2. Paths and configuration.
      3. Secrets: rotate first (write the instruction down), then remove, then .gitignore.
      4. Dependencies: what does the code import that requirements.txt does not name?
      5. The README: install AND run, plus what the reader should see.
      6. The notebook: clear the counts, then execute it. Never renumber by hand.
    """
    pass


fix_survival_baseline(BASELINE)

score = audit(BASELINE, "survival-baseline, after your repair")
ok, msg = restart_and_run_all(BASELINE / "train.ipynb", cwd=BASELINE)
print("train.ipynb Restart-and-Run-All:", "PASSED" if ok else "FAILED")
print("   ", msg.strip().splitlines()[-1][:200] if msg.strip() else "(no output)")
print()
print(f"Finished when this line reads 6 of 6 and PASSED. Right now: {score} of 6.")

AUDIT  survival-baseline, after your repair
FAIL  G1 README
        - README.md: no run command (jupyter lab / python x.py / make ...)
FAIL  G2 dependencies
        - imported but not declared in requirements.txt: numpy
        - imported but not declared in requirements.txt: scikit-learn
PASS  G3 paths
FAIL  G4 data
        - train.ipynb[cell 2]: opens 'data/titanic.csv', which is neither in the repository nor documented
FAIL  G5 notebooks
        - train.ipynb: execution counts are [1, 2, 5], not [1, 2, 3]
FAIL  G6 secrets
        - train.ipynb[cell 1]: WANDB_API_KEY = "tvtc-lab-1111-EXAMPLE-not-a-real-credential"
        - no .gitignore, so a future .env or credentials file gets committed by accident
1 of 6 gates pass



train.ipynb Restart-and-Run-All: FAILED
    CellExecutionError: FileNotFoundError: [Errno 2] No such file or directory: 'data/titanic.csv'

Finished when this line reads 6 of 6 and PASSED. Right now: 1 of 6.


## 💬 Discuss

Take these to the group. None of them has a lookup answer, and two of them are arguments people
have in real teams every week.

1. Section 6 showed a repository that passes a secrets gate while a live credential sits in its
   history. Your team's CI runs that gate on every pull request and it has been green for a year.
   What is that green light actually evidence of? Write the sentence you would put in a handover
   document to describe, honestly, what the check covers.
2. The audit in this lesson has six gates. Adding a seventh costs you nothing today and costs
   every future contributor a little time forever. Propose the seventh gate you would add to a
   student project repository, then argue the case *against* it as if you were the person it will
   annoy. Which side wins, and what changed your mind?
3. `IMPORT_TO_PACKAGE` in section 3 is a hand-written map of eleven special cases. It will be
   wrong the first time somebody imports something not in it. Three options: keep extending the
   map, drop the check entirely, or replace it with `pip freeze` from a clean environment. Each
   fails differently. Pick one and name the failure you are accepting.
4. Pimentel et al. found 36.36% of notebooks have out-of-order cells and only 4.03% reproduce
   their own outputs. Section 5 argued the counters are *evidence* and must never be hand-edited.
   But a notebook you are still exploring in is genuinely out of order all day, and that is not a
   defect — it is how the tool works. Where exactly is the line between "still working" and
   "handing over", and what should happen at that moment?
5. Which of the six gates would have saved you the most time in the last month of your own work?
   Be specific: name the incident.

## Summary

A repository someone else can use is a repository that answers five questions before it is asked:
what is this, how do I install it, how do I run it, where is the data, and what will I see. Six
mechanical gates cover those questions — README, dependencies, paths, data, notebook execution,
secrets — and all six can be checked offline, in seconds, by code you now have.

The static half is cheap and finds most of it. The execution half — Restart Kernel and Run All, in
a fresh kernel, from the directory the reader will be in — is the only one that cannot be satisfied
by tidying a file, and it is the one that Pimentel's 4.03% says almost nobody performs.

Two limits are not incidental, they are the lesson. A gate reports on what it read, so a secrets
gate that reads the working tree says nothing about the history, and a leaked credential is fixed
by rotating it, not by deleting it. And every gate you will ever meet was written after somebody
had the defect it catches — including, five times over and with the dates in the commit log, the
gates in this repository.

## Self-check

1. **A colleague sends you a repository whose CI badge is green and whose secrets scan passes.
   What have you learned about the repository?** Look again at what the G6 implementation in
   section 3 actually opens, and at what section 6 found afterwards.
2. **A notebook's execution counts read `1, 2, 3, 4`. What does that prove?** Compare what the
   repair in section 4 wrote into the file against what the execution in section 5 wrote into it.
3. **Why does `requirements.txt` exist if `pip freeze` also works?** Look at what the generated
   file in section 4 contains, and at the failure rate Pimentel et al. measured for
   `requirements.txt` against `setup.py`.
4. **You find an absolute path in a notebook. Name two fixes, and say which one you would use for
   a 4 GB dataset.** Section 4 lists three endings for the data story; only one of them survives
   the file being too big to commit.
5. **Your teammate hand-edits execution counts to `1..n` so the audit passes. The audit now passes.
   What did they break?** Say what the counter was evidence of, and what it is evidence of now.

## ⚠️ Where this breaks

- **This entire strand has no outcome evidence, and you should know that before you spend twelve
  hours on it.** There is no study showing that teaching shell, git, environments and handover
  makes graduates measurably better. MIT's *Missing Semester* — nine lectures on exactly this
  material, on the grounds that it is "rarely covered and is instead left for students to pick up
  on their own" — publishes no enrolment or effectiveness figures. The argument for this strand is
  a *prerequisite* argument, not an evidenced intervention: peer review of real work, written
  feedback inside artefacts, and being handed a repository all presuppose these skills, and none
  of them is deliverable without them. That is an honest reason to teach it. It is not the same
  kind of reason as the *g* = 0.31 behind peer assessment, and it should not be dressed up as one.
- **Six gates is a choice, not a standard.** Nothing here checks tests, licences, code style, type
  hints, CI configuration, model cards, reproducible seeds, or whether the analysis is correct. A
  repository can pass all six and be worthless. The gates buy you the first ten minutes; they buy
  nothing after that.
- **`IMPORT_TO_PACKAGE` is eleven hand-written special cases** and there are thousands of packages
  whose import name differs from their distribution name. G2 will report a false "missing"
  whenever it meets one it does not know. The industrial answer is a lock file produced by the
  resolver itself (`pip freeze`, `uv pip compile`, `poetry.lock`), not a lookup table.
- **The absolute-path regex only knows four shapes.** `/Users/`, `/home/`, `/Volumes/`, and a
  drive letter followed by a doubled backslash or a forward slash. A path held in a variable, a
  path built by `os.path.join`, a path read from a config file, a network share written
  `\\server\share`, and a raw-string Windows path such as `r"C:\Users\nora"` — single backslashes —
  all pass straight through it.
- **That last exclusion is not laziness, it is a scar, and you can see it in the cell you ran.**
  The first version of this pattern accepted a *single* backslash after the drive letter. Run
  over the 420 notebooks in section 7 it reported one absolute path, in
  `Course 06/unit1-ethics-foundations/examples/03_case_study_analysis.ipynb`. The offending code
  was a chart label: `"Equal predictive\nvalue (Northpointe's:\n'is a 7 a 7 for all?')"`. The
  apostrophe in *Northpointe's*, followed by `s`, a colon and an escaped newline, is
  character-for-character a quote, a drive letter and a backslash. Requiring the doubled
  backslash that a real Windows path must have inside a Python string removes the false positive
  and costs the raw-string case above. One false positive in 420 notebooks is enough to teach
  everyone to ignore the checker, which is worse than not having one — the same reasoning the
  commit for `check_key_rubric_agreement.py` gives for its three-line lookback: *"a gate that
  cries wolf trains everyone to ignore it."*
- **`restart_and_run_all` proves the notebook ran on *this* machine, today.** It says nothing about
  a machine without your packages, without your GPU, in a different timezone, or after the upstream
  dataset URL rots. Running it in CI on a clean container is the version of this check that means
  something, and it costs money and setup this notebook deliberately does not require.
- **The scratch repositories are deleted at the end.** Every commit, every fix and the exercise
  repository disappear when the cleanup cell runs. Set `CLEAN_UP = False` in that cell if you want
  to keep working in `LAB`. Nothing in this lesson ever writes to the AI Diploma repository, and
  the assertion in the setup cell is what enforces that rather than good intentions.
- **The git history read in section 7 is this repository's, at this moment.** Re-run it next month
  and the log will be longer. The dated facts quoted in the prose are from commit messages you can
  read yourself with `git log -1 --format=%B <commit>`; if a line in the printed log ever
  contradicts the prose, believe the log.

## 📚 References

1. Pimentel, J. F., Murta, L., Braganholo, V., & Freire, J. (2019). *A Large-scale Study about
   Quality and Reproducibility of Jupyter Notebooks*. Proceedings of the 16th International
   Conference on Mining Software Repositories (MSR '19), 507–517.
   <https://leomurta.github.io/papers/pimentel2019a.pdf> — source of the 1,159,166-notebook corpus,
   the 24.11% / 4.03% execution figures, the 36.36% out-of-order figure, and the dependency-file
   installation failure rates.
2. Meli, M., McNiece, M. R., & Reaves, B. (2019). *How Bad Can It Git? Characterizing Secret
   Leakage in Public GitHub Repositories*. Network and Distributed System Security Symposium
   (NDSS 2019). <https://www.ndss-symposium.org/ndss-paper/how-bad-can-it-git-characterizing-secret-leakage-in-public-github-repositories/>
3. Double, K. S., McGrane, J. A., & Hopfenbeck, T. N. (2020). *The Impact of Peer Assessment on
   Academic Performance: A Meta-Analysis of Control Group Studies*. Educational Psychology Review,
   32(2), 481–509. <https://eric.ed.gov/?id=EJ1255661> — *g* = 0.31 overall (54 studies, k = 141);
   *g* = 0.28 versus teacher assessment. Note the honest limit: the corpus is largely writing and
   general coursework, and almost none of it assesses code.
4. MIT CSAIL. *The Missing Semester of Your CS Education*. <https://missing.csail.mit.edu/about/>
   — nine lectures on "computing ecosystem literacy", the material this strand covers. The page
   publishes no outcome data, which is why the strand is argued for as a prerequisite rather than
   as an evidenced intervention.
5. This repository: `tools/verify/` (the gates), `tools/data.py` and `Course 04/datasets/DATA.md`
   (the data story), and `git log -- tools/verify` (the history read in section 7).

In [16]:
# WHAT: delete the scratch laboratory, after checking one more time that it is not the repository.
# WHY:  a lesson that leaves half-built git repositories on your disk is exactly the kind of mess
#       this lesson is about. Set CLEAN_UP = False if you want to keep working inside LAB.
CLEAN_UP = True

if CLEAN_UP:
    if REPO is not None:
        assert REPO != LAB and REPO not in LAB.parents, "refusing to delete: LAB is inside the repo"
    assert LAB.name.startswith("handover_lab_"), f"refusing to delete an unexpected path: {LAB}"
    shutil.rmtree(LAB, ignore_errors=True)
    print("removed:", LAB)
    print("still exists:", LAB.exists())
else:
    print("kept:", LAB)

removed: /private/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/handover_lab_x3wapqz4
still exists: False
